# Prompt → LaTeX Inserts

Reads pipeline prompts from the `historical_norms` and `grpo_training` dagspaces
and formats them as `\begin{promptbox}` LaTeX environments for manuscript appendices.

**Outputs**: One `.tex` file per prompt, plus a combined `all_prompts.tex` with `\input{}` stubs.

In [ ]:
import yaml
import textwrap
from pathlib import Path
from IPython.display import display, Markdown

# ── Paths ──
REPO = Path("/share/pierson/matt/UAIR")
HN_PROMPTS = REPO / "dagspaces/historical_norms/conf/prompt"
GRPO_PROMPTS = REPO / "dagspaces/grpo_training/conf/prompt"

OUT_DIR = Path("tables/prompts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PAPER_DIR = REPO / "papers/colm26_normative-simulacra/prompts"
PAPER_DIR.mkdir(parents=True, exist_ok=True)

def load_prompt(path: Path) -> dict:
    with open(path) as f:
        return yaml.safe_load(f)

## 1. Define which prompts to include

In [ ]:
# ── Load SFT constants directly from the source module ──
import re as _re

def _load_sft_constants():
    src_path = REPO / "dagspaces/grpo_training/stages/sft_data_prep.py"
    source = src_path.read_text()
    ns = {}
    for name in ['_CI_INSTRUCTION', '_NO_EXCHANGE_REASONING']:
        match = _re.search(rf'^{name}\s*=\s*\((.+?)^\)', source, _re.MULTILINE | _re.DOTALL)
        if match:
            ns[name] = eval(f'({match.group(1)})')
    return ns

sft_consts = _load_sft_constants()
_CI_INSTRUCTION = sft_consts['_CI_INSTRUCTION']
_NO_EXCHANGE_REASONING = sft_consts['_NO_EXCHANGE_REASONING']
print(f'Loaded SFT constants: _CI_INSTRUCTION ({len(_CI_INSTRUCTION)} chars), '
      f'_NO_EXCHANGE_REASONING ({len(_NO_EXCHANGE_REASONING)} chars)')

# ── SFT completion schema ──
_SFT_SCHEMA = '{\n'\
    '  "reasoning": "<narrative trace covering all flows>",\n'\
    '  "has_information_exchange": true,\n'\
    '  "flows": [\n'\
    '    {\n'\
    '      "sender": "...",\n'\
    '      "recipient": "...",\n'\
    '      "subject": "...",\n'\
    '      "information_type": "...",\n'\
    '      "transmission_principle": "...",\n'\
    '      "context": "...",\n'\
    '      "appropriateness": "appropriate | inappropriate | ambiguous",\n'\
    '      "norms_invoked": ["..."],\n'\
    '      "norm_source": "explicit | implicit | both",\n'\
    '      "is_new_flow": false,\n'\
    '      "confidence": 8\n'\
    '    }\n'\
    '  ]\n'\
    '}'

_SFT_NEG_SCHEMA = '{\n'\
    '  "reasoning": "<reason why no information exchange occurs>",\n'\
    '  "has_information_exchange": false,\n'\
    '  "flows": []\n'\
    '}'

# ── Manifest ──
PROMPT_MANIFEST = [
    # Gold-standard extraction pipeline (historical_norms)
    ("norm-reasoning-fiction",
     HN_PROMPTS / "norm_reasoning_fiction.yaml",
     "Norm reasoning prompt for fiction passages (Stage 1 of norm extraction)."),
    ("norm-extraction-fiction",
     HN_PROMPTS / "norm_extraction_fiction.yaml",
     "Structured norm extraction from fiction (Stage 2)."),
    ("ci-reasoning-fiction",
     HN_PROMPTS / "ci_reasoning_fiction.yaml",
     "CI flow reasoning prompt for fiction passages (Stage 1 of flow extraction)."),
    ("ci-extraction-fiction",
     HN_PROMPTS / "ci_extraction_fiction.yaml",
     "Structured CI flow extraction from fiction (Stage 2)."),
    ("norm-role-abstraction",
     HN_PROMPTS / "norm_role_abstraction.yaml",
     "Character-to-role abstraction for extracted norms."),
    ("norm-consolidation",
     HN_PROMPTS / "norm_consolidation.yaml",
     "Semantic deduplication and consolidation of extracted norms."),
    # SFT data templating (from sft_data_prep.py)
    ("sft-user-instruction",
     {"system_prompt": "",
      "prompt_template": _CI_INSTRUCTION + "\n\n{{article_text}}"},
     "SFT user-turn instruction prepended to each fiction passage."),
    ("sft-completion-schema",
     {"system_prompt": "Positive example (has_information_exchange = true):",
      "prompt_template": _SFT_SCHEMA},
     "SFT assistant-turn completion schema (JSON). Each flow is a flat CI tuple with contextual metadata."),
    ("sft-negative-example",
     {"system_prompt": "Negative example (has_information_exchange = false):",
      "prompt_template": _SFT_NEG_SCHEMA + "\n\nDefault reasoning when no trace available:\n" + _NO_EXCHANGE_REASONING},
     "SFT negative example template. Capped at 1:1 ratio with positives."),
    # GRPO training prompts (YAML)
    ("grpo-ci-extraction",
     GRPO_PROMPTS / "ci_extraction.yaml",
     "CI flow extraction instruction used for GRPO online inference."),
    ("grpo-reward-judge",
     GRPO_PROMPTS / "reward_judge.yaml",
     "Normative grounding reward judge for GRPO training."),
    ("grpo-no-flow-judge",
     GRPO_PROMPTS / "no_flow_judge.yaml",
     "Coverage judge for no-flow predictions in GRPO reward."),
    ("grpo-norm-judgment",
     GRPO_PROMPTS / "norm_judgment.yaml",
     "Norm application judgment prompt (CIRL vignette evaluation)."),
]

print(f'\n{len(PROMPT_MANIFEST)} prompts to format')
for label, source, desc in PROMPT_MANIFEST:
    if isinstance(source, Path):
        assert source.exists(), f'Missing: {source}'
        print(f'  {label:30s} {source.name}')
    else:
        print(f'  {label:30s} [inline]')


## 2. LaTeX formatting

In [ ]:
def _escape_latex(text: str) -> str:
    """Escape special LaTeX characters in prompt text."""
    replacements = [
        ("\\", r"\textbackslash{}"),
        ("&", r"\&"),
        ("%", r"\%"),
        ("$", r"\$"),
        ("#", r"\#"),
        ("_", r"\_"),
        ("{", r"\{"),
        ("}", r"\}"),
        ("~", r"\textasciitilde{}"),
        ("^", r"\textasciicircum{}"),
    ]
    for old, new in replacements:
        text = text.replace(old, new)
    # Preserve template variables: turn escaped \{\{var\}\} back to {{var}}
    import re
    text = re.sub(r'\\\{\\\{(.*?)\\\}\\\}', r'{{\1}}', text)
    return text


def prompt_to_latex(label: str, data: dict, description: str) -> str:
    """Convert a prompt YAML dict to a LaTeX promptbox.
    
    Expects either (system_prompt, prompt_template) or (instruction, prompt_template).
    Uses a tcolorbox-based promptbox environment.
    """
    system = data.get("system_prompt", data.get("instruction", ""))
    template = data.get("prompt_template", "")

    lines = []
    lines.append(f"% Auto-generated from {label}")
    lines.append(f"\\begin{{promptbox}}{{{_escape_latex(description)}}}")
    lines.append(f"\\label{{prompt:{label}}}")
    lines.append("")

    # System prompt
    if system.strip():
        lines.append(r"\textbf{System Prompt:}")
        lines.append(r"\begin{prompttext}")
        lines.append(_escape_latex(system.strip()))
        lines.append(r"\end{prompttext}")
        lines.append("")

    # User template
    if template.strip():
        lines.append(r"\textbf{User Template:}")
        lines.append(r"\begin{prompttext}")
        lines.append(_escape_latex(template.strip()))
        lines.append(r"\end{prompttext}")

    lines.append(r"\end{promptbox}")
    lines.append("")
    return "\n".join(lines)


print("Formatting functions defined.")

## 3. Generate and save LaTeX files

In [ ]:
all_latex_parts = []
all_latex_parts.append('% ' + '=' * 70)
all_latex_parts.append('% Pipeline prompts -- auto-generated by prompt_to_latex.ipynb')
all_latex_parts.append('% ' + '=' * 70)
all_latex_parts.append('')

sections = {
    "norm-reasoning-fiction": r"\subsection{Gold-Standard Norm \& Flow Extraction}",
    "sft-user-instruction": r"\subsection{SFT Data Templating}",
    "grpo-ci-extraction": r"\subsection{GRPO Training Prompts}",
}

for label, source, desc in PROMPT_MANIFEST:
    if isinstance(source, Path):
        data = load_prompt(source)
    else:
        data = source

    tex = prompt_to_latex(label, data, desc)

    out_path = OUT_DIR / f'{label}.tex'
    out_path.write_text(tex)
    paper_path = PAPER_DIR / f'{label}.tex'
    paper_path.write_text(tex)

    if label in sections:
        all_latex_parts.append(sections[label])
        all_latex_parts.append('')

    all_latex_parts.append(tex)

    sys_text = data.get('system_prompt', data.get('instruction', ''))
    tmpl_text = data.get('prompt_template', '')
    print(f'  {label:30s}  sys={len(sys_text):>5d} chars  tmpl={len(tmpl_text):>4d} chars  -> {out_path.name}')

combined = '\n'.join(all_latex_parts)
(OUT_DIR / 'all_prompts.tex').write_text(combined)
(PAPER_DIR / 'all_prompts.tex').write_text(combined)

print(f'\nSaved {len(PROMPT_MANIFEST)} individual .tex files + all_prompts.tex')
print(f'  -> {OUT_DIR}')
print(f'  -> {PAPER_DIR}')


## 4. Preview a sample prompt

In [ ]:
# Show the SFT user instruction as an example
sample_data = {'system_prompt': '',
               'prompt_template': _CI_INSTRUCTION + '\n\n{{article_text}}'}
print(prompt_to_latex('sft-user-instruction', sample_data,
                      'SFT user-turn instruction prepended to each fiction passage.'))


## 5. LaTeX preamble snippet

Add this to your manuscript preamble to define the `promptbox` and `prompttext` environments:

```latex
\usepackage{tcolorbox}
\tcbuselibrary{breakable, skins}

\newtcolorbox{promptbox}[1]{
  colback=gray!5, colframe=gray!50,
  fonttitle=\bfseries\small, title={#1},
  breakable, enhanced,
  left=4pt, right=4pt, top=4pt, bottom=4pt,
  boxrule=0.5pt,
}

\newenvironment{prompttext}{%
  \begin{quote}\ttfamily\small\raggedright
}{%
  \end{quote}
}
```